# 联合校准与耦合验证

独立、可丢弃的合成 sandbox，无设备，无前置课程结果。建议先了解单目标“参数校准与恢复”的概念。源码在 `src/my_experiment/joint_calibration.py`，复用的单目标模型与分析在旁边的 `calibration.py`。

本课刻意展示：**修改不同参数行没有冲突，也可能相互影响；单独通过不代表组合通过。**

In [ ]:
import scopecat as sc

session = sc.notebook()
session

In [ ]:
from uuid import uuid4
from my_experiment.calibration import Sensor
from my_experiment.joint_calibration import JointIntent, calibrate_joint

assert session.project_root is not None
project = sc.open_project(session.project_root)
lab = project.connect()
trial = "joint-" + uuid4().hex[:8]
initial = lab.parameters.save(
    name=trial + "-initial", catalog=sc.parameter_catalog("sensors", Sensor),
    parameters=sc.parameter_snapshot("joint-inputs", tables={Sensor: [
        Sensor(id="q0", offset=0.1), Sensor(id="q1", offset=0.1),
    ]}),
)
destination = lab.parameters.create_branch(trial, revision=initial)
print("本次参数分支:", trial)

## 先观察没有耦合的情况

每个通道的零点是 0.24，初始设置为 0.1。流程先分别拟合、分别验证，再合并两个候选。此时不能发布，必须用合并后的参数重新测量两个通道。

下面在合并后暂停：已保留八个采集/分析步骤（含四次采集），加上一个 compose 步骤。

In [ ]:
request = lab.procedures.submit(
    calibrate_joint,
    JointIntent(initial=lab.parameters.resolve(initial), destination=destination,
                revision_name=trial + "-accepted", coupling=0.0),
    request_key=trial,
)
lab.procedures.resume_snapshot(
    request.snapshot, should_yield=lambda: len(request.steps().items) >= 9,
)
request_id = request.id
assert request.state == "ready"
assert lab.parameters.checkout(trial).head == destination
assert len(request.steps().items) == 9
request.summary()

## 从联合候选恢复

关闭客户端再重连。保留的候选和步骤会重放，不重复拟合和单独验证。自动回归还会在这个暂停点重启 daemon。

In [ ]:
lab.close()
lab = project.connect()
request = lab.procedures.get(request_id)
request.resume()
assert request.summary().outcome == "succeeded"
assert len(request.steps().items) == 13
accepted = lab.parameters.checkout(trial).head
assert accepted.generation == destination.generation + 1
request.resume()
assert lab.parameters.checkout(trial).head == accepted
request.summary()

## 单独都通过，联合却失败

合成模型加入 `coupling * (own - 0.1) * (peer - 0.1)`。只有一行改变时，另一行仍为 0.1，耦合项为零；两行都改到 0.24 时，耦合项成为 0.0196，再加 0.002 的验证扰动，超过 0.01 阈值。

这不是量子器件物理模型，而是一个可复现的反例。我们从同一个初始版本创建独立分支，不污染已经接受的那一支。

In [ ]:
rejected_branch = lab.parameters.create_branch(trial + "-coupled", revision=initial)
rejected = lab.procedures.submit(
    calibrate_joint,
    JointIntent(initial=lab.parameters.resolve(initial), destination=rejected_branch,
                revision_name=trial + "-not-published", coupling=1.0),
    request_key=trial + "-coupled",
)
try:
    rejected.resume()
except ValueError as error:
    print(error)
assert rejected.summary().outcome == "failed"
assert lab.parameters.checkout(trial + "-coupled").head == rejected_branch
for target in ("q0", "q1"):
    analysis = lab.published_analysis(rejected.output(f"individual-decision-{target}").analysis_record_id)
    assert analysis.fact("decision").value["accepted"] is True
verification = lab.published_analysis(rejected.output("verify-joint").analysis_record_id)
decision = verification.fact("decision").value
assert decision["accepted"] is False
assert set(decision["rejected"]) == {"q0", "q1"}
assert not decision["missing"]
print(decision)
rejected.summary()

## 完整覆盖也必须验证

下面故意漏掉 q1 的联合检查。策略仍保留已有输入，但 missing 中出现 q1，accepted 必须为 false。此决策不能授权发布隐含的子集。

In [ ]:
from my_experiment.joint_calibration import verify_joint

incomplete = lab.analyze(verify_joint(
    targets=("q0", "q1"),
    baselines={target: lab.get_run(request.output(f"baseline-{target}").run_id) for target in ("q0", "q1")},
    checks={"q0": lab.get_run(request.output("joint-q0").run_id)},
    tolerance=0.01,
)).fact("decision").value
assert incomplete["accepted"] is False
assert not incomplete["rejected"]
assert tuple(incomplete["missing"]) == ("q1",)
incomplete

## 改一个地方再观察

- 把 coupling 改为 0.2，在新请求中观察联合残差。不要通过放宽阈值把这个练习当作真实科学结论。
- 对照源码中的 `combine_parameter_candidates`、联合采集、`verify_joint` 和 `publish_parameter_candidate`：合并负责数值冲突，策略负责科学覆盖，发布负责精确版本与证据事务。
- 重启 kernel 后用 `lab.procedures.list()` 找到原请求。修改 procedure 源码时升级 version 并提交新请求，不要改写正在恢复的意图。
- 当前实验是顺序执行；本课不引入自动 freshness、目标选择或并发调度。真实多目标模型还要明确共享设备、测量条件以及哪些交互必须观察。

已完成的分支和失败证据都保留在本 sandbox。需要从零开始时使用重置，旧副本不会自动删除。

In [ ]:
lab.close()